# Репликационный эксперимент: SGD, хвостовой показатель и разреживание

Ноутбук запускает репликационный эксперимент для сети FCN4 на MNIST:
- обучение стохастическим градиентным спуском при разных значениях шага обучения и размера мини-батча;
- оценка хвостового показателя весовых коэффициентов;
- построение графика зависимости хвостового показателя от отношения шага обучения к размеру мини-батча;
- применение четырёх способов разреживания обученных моделей.

Код сохраняет результаты, графики и контрольные точки модели в отдельные папки.


In [ ]:
# Подготовка окружения: библиотеки, фиксирование случайности и папки результатов

import os
import math
import copy
import json
import random
import time
import zipfile
import glob
from dataclasses import dataclass, asdict
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as transforms

SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

RESULTS_DIR = "results"
FIGURES_DIR = "figures"
CHECKPOINTS_DIR = "checkpoints"

for d in [RESULTS_DIR, FIGURES_DIR, CHECKPOINTS_DIR]:
    os.makedirs(d, exist_ok=True)

In [ ]:
# Загрузка MNIST и определение архитектуры FCN4

def get_dataloaders(dataset="MNIST", batch_size=128, shuffle_train=True):
    dataset = dataset.upper()

    if dataset != "MNIST":
        raise ValueError("В этих ноутбуках настроен эксперимент только для MNIST.")

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])

    train_set = torchvision.datasets.MNIST(
        root="./data", train=True, download=True, transform=transform
    )
    test_set = torchvision.datasets.MNIST(
        root="./data", train=False, download=True, transform=transform
    )

    train_loader = DataLoader(
        train_set,
        batch_size=batch_size,
        shuffle=shuffle_train,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    test_loader = DataLoader(
        test_set,
        batch_size=256,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    input_dim = 28 * 28
    num_classes = 10

    return train_loader, test_loader, input_dim, num_classes


class FCN4(nn.Module):
    """
    FCN4 для MNIST:
    4 скрытых полносвязных слоя ширины width + выходной слой.
    """
    def __init__(self, input_dim=784, width=2048, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, width),
            nn.ReLU(),
            nn.Linear(width, width),
            nn.ReLU(),
            nn.Linear(width, width),
            nn.ReLU(),
            nn.Linear(width, width),
            nn.ReLU(),
            nn.Linear(width, num_classes),
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.net(x)


def build_fcn4(width=2048):
    return FCN4(input_dim=784, width=width, num_classes=10)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def evaluate(model, loader):
    model.eval()

    total_loss = 0.0
    total = 0
    correct = 0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            logits = model(x)
            loss = F.cross_entropy(logits, y)

            bs = x.size(0)
            total_loss += loss.item() * bs
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += bs

    return {
        "loss": total_loss / total,
        "acc": correct / total
    }

In [ ]:
# Оценивание хвостового показателя весовых коэффициентов

def find_m_barsbey(N, m=0):
    """Подбирает допустимый делитель для MMO-оценки."""
    if m == 0:
        m = int(np.sqrt(N))
    m = max(2, int(m))

    while m > 1 and N % m != 0:
        m -= 1

    return m


def alpha_estimator_multi_barsbey(m, X):
    """Вычисляет многомерную MMO-оценку хвостового показателя."""
    if not torch.is_tensor(X):
        X = torch.from_numpy(np.asarray(X, dtype=np.float64))

    X = X.to(dtype=torch.float64, device="cpu")

    N = X.shape[0]
    if m <= 1 or N < 2 * m or N % m != 0:
        return np.nan

    n = int(N / m)

    try:
        Y = torch.sum(X.view(n, m, -1), dim=1)
    except RuntimeError:
        return np.nan

    eps = np.spacing(1)

    Y_log_norm = torch.log(torch.linalg.norm(Y, dim=1) + eps).mean()
    X_log_norm = torch.log(torch.linalg.norm(X, dim=1) + eps).mean()

    diff = (Y_log_norm - X_log_norm) / math.log(m)

    if (not torch.isfinite(diff)) or diff.item() <= 0:
        return np.nan

    alpha = 1.0 / diff.item()

    if not np.isfinite(alpha):
        return np.nan

    return float(alpha)


# Оценка хвостового показателя для одной матрицы весовых коэффициентов.
def est_alpha_barsbey_matrix(X, clip_to_2=True):
    """
    Оценка alpha для одного слоя в стиле кода Barsbey et al.

    X — матрица весов слоя размера N x d.
    Кандидаты для параметра агрегации m берутся из:
        2, 5, 10, 20, 50, 100, 500, 1000,
    если они подходят по размеру N. Также добавляется m около sqrt(N).
    Итоговая оценка — медиана по всем допустимым m.
    """
    X = np.asarray(X, dtype=np.float64)

    if X.ndim != 2:
        X = X.reshape(X.shape[0], -1)

    X = X[np.all(np.isfinite(X), axis=1)]

    N, d = X.shape
    if N < 4 or d < 1:
        return np.nan

    m0 = find_m_barsbey(N)

    if N == 10:
        candidates = [2, 5]
    else:
        base_candidates = [k for k in [2, 5, 10, 20, 50, 100, 500, 1000] if k <= N / 2]
        candidates = []
        for k in base_candidates:
            candidates.append(k if N % k == 0 else find_m_barsbey(N, k))

    candidates = sorted(set([m for m in candidates if m > 1 and m <= N // 2 and N % m == 0]))

    if m0 > 1 and m0 <= N // 2 and N % m0 == 0 and m0 not in candidates:
        candidates.append(m0)

    candidates = sorted(set(candidates))

    if len(candidates) == 0:
        return np.nan

    X_torch = torch.from_numpy(X.astype(np.float64, copy=False))

    estimates = []
    for m in candidates:
        a = alpha_estimator_multi_barsbey(m, X_torch)
        if np.isfinite(a):
            estimates.append(a)

    if len(estimates) == 0:
        return np.nan

    alpha = float(np.median(estimates))

    if clip_to_2:
        alpha = min(alpha, 2.0)

    return alpha


# Повторяет оценивание для слоя при нескольких перестановках строк.
def estimate_alpha_layer_barsbey_style(
    weight_array,
    n_repeats=5,
    random_state=42,
    clip_to_2=True,
    transpose_last_layer=False
):
    """
    Оценка хвостового показателя для одного слоя максимально близко к
    реализации Barsbey et al.

    Шаги:
    1. слой рассматривается как матрица;
    2. для последнего слоя FCN применяется транспонирование, как в их dream_team=True;
    3. веса центрируются по медиане столбцов;
    4. строки матрицы перемешиваются;
    5. MMO-оценка считается несколько раз;
    6. итоговая оценка — медиана по повторам.
    """
    a = np.asarray(weight_array, dtype=np.float64)

    if a.ndim == 4:
        a = np.reshape(a, (np.prod(a.shape[:2]), np.prod(a.shape[2:])))
    elif a.ndim == 2:
        pass
    else:
        a = a.reshape(a.shape[0], -1)

    if transpose_last_layer:
        a = a.T

    if a.shape[0] < 4:
        return np.nan

    # Центрирование по столбцам, как в alpha_estimation из репозитория.
    a = a - np.median(a, axis=0)

    rng = np.random.default_rng(random_state)

    estimates = []
    for _ in range(n_repeats):
        perm_idx = rng.permutation(a.shape[0])
        a_perm = a[perm_idx, :]
        alpha = est_alpha_barsbey_matrix(a_perm, clip_to_2=clip_to_2)

        if np.isfinite(alpha):
            estimates.append(alpha)

    if len(estimates) == 0:
        return np.nan

    return float(np.median(estimates))


def estimate_model_tail_indices_mmo(model, n_repeats=5, clip_to_2=True, skip_bias=True):
    """
    Оценка alpha по всем весовым слоям модели.

    Возвращает:
    - alpha_df: alpha по каждому слою;
    - mean_alpha: среднее alpha по слоям.

    Для прямого сопоставления с Figure 1 Barsbey et al. используется mean_alpha.
    """
    weight_items = [
        (name, p)
        for name, p in model.named_parameters()
        if (not skip_bias) or ("weight" in name)
    ]

    rows = []

    for idx, (name, p) in enumerate(weight_items):
        if "weight" not in name:
            continue

        W = p.detach().cpu().numpy()

        # В коде Barsbey et al. при dream_team=True последний слой транспонируется.
        is_last_weight_layer = (idx == len(weight_items) - 1)

        alpha_hat = estimate_alpha_layer_barsbey_style(
            W,
            n_repeats=n_repeats,
            random_state=42 + idx,
            clip_to_2=clip_to_2,
            transpose_last_layer=is_last_weight_layer
        )

        rows.append({
            "layer": name,
            "n_params": int(W.size),
            "matrix_shape": str(tuple(W.shape)),
            "transposed_for_alpha": bool(is_last_weight_layer),
            "alpha_hat": alpha_hat
        })

    df = pd.DataFrame(rows)

    valid = df["alpha_hat"].dropna()
    mean_alpha = np.nan if len(valid) == 0 else float(valid.mean())

    return df, mean_alpha


def mean_alpha_prunable_layers_from_df(alpha_df, skip_last=True):
    """
    Среднее alpha только по слоям, участвующим в pruning.

    Если skip_last=True, последний классификационный слой исключается.
    """
    df = alpha_df.copy()
    df = df[df["alpha_hat"].notna()].reset_index(drop=True)

    if len(df) == 0:
        return np.nan

    if skip_last and len(df) > 1:
        df = df.iloc[:-1].copy()

    if len(df) == 0:
        return np.nan

    return float(df["alpha_hat"].mean())


In [ ]:
# Обучение FCN4 стохастическим градиентным спуском

@dataclass
class SGDConfig:
    dataset: str = "MNIST"
    model_name: str = "FCN4"
    width: int = 2048
    lr: float = 0.30
    batch_size: int = 100
    seed: int = 42


def train_sgd_article_style(
    config: SGDConfig,
    max_epochs=80,
    convergence_train_acc=0.99,
    convergence_loss=0.03,
    avg_steps=1000,
    verbose=True
):
    """Обучает модель SGD без момента и затем усредняет параметры на дополнительных шагах."""

    set_seed(config.seed)

    train_loader, test_loader, _, _ = get_dataloaders(
        dataset=config.dataset,
        batch_size=config.batch_size,
        shuffle_train=True
    )

    model = build_fcn4(width=config.width).to(device)

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=config.lr,
        momentum=0.0
    )

    history = []
    start_time = time.time()

    converged = False
    stopped_by_nan = False

    last_valid_epoch = None
    last_valid_row = None
    last_valid_state_dict = None

    for epoch in range(1, max_epochs + 1):
        model.train()

        total_loss = 0.0
        total = 0
        correct = 0

        pbar = tqdm(train_loader, desc=f"SGD epoch {epoch}/{max_epochs}", leave=False)

        for x, y in pbar:
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = F.cross_entropy(logits, y)

            if not torch.isfinite(loss):
                stopped_by_nan = True
                print("NaN/Inf loss detected inside epoch. Stopping this run.")
                break

            loss.backward()
            optimizer.step()

            bs = x.size(0)
            total_loss += loss.item() * bs
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += bs

            pbar.set_postfix(loss=total_loss / max(total, 1), acc=correct / max(total, 1))

        if stopped_by_nan:
            break

        train_loss = total_loss / total
        train_acc = correct / total
        test_metrics = evaluate(model, test_loader)

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "test_loss": test_metrics["loss"],
            "test_acc": test_metrics["acc"]
        }
        history.append(row)

        if verbose:
            print(row)

        if (
            not np.isfinite(train_loss)
            or not np.isfinite(train_acc)
            or not np.isfinite(test_metrics["loss"])
            or not np.isfinite(test_metrics["acc"])
        ):
            stopped_by_nan = True
            print("NaN/Inf detected after epoch. Stopping this run.")
            break

        # Сохраняем последнюю корректную модель.
        # Это НЕ выбор по test accuracy.
        last_valid_epoch = epoch
        last_valid_row = row.copy()
        last_valid_state_dict = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }

        if train_acc >= convergence_train_acc and train_loss <= convergence_loss:
            converged = True
            if verbose:
                print(
                    f"Article-style convergence criterion reached at epoch {epoch}: "
                    f"train_acc={train_acc:.4f}, train_loss={train_loss:.6f}"
                )
            break

    if last_valid_state_dict is not None:
        model.load_state_dict(last_valid_state_dict)
        if verbose:
            print(f"Restored last valid model from epoch {last_valid_epoch}")
    else:
        print("Warning: no valid model was saved before stopping.")

    # Усреднение SGD-шагов после остановки.
    # Как и в статье, это делается после достижения критерия сходимости.
    # Если при усреднении появляется NaN/Inf, усреднение останавливается.
    avg_state = {
        name: torch.zeros_like(p.data)
        for name, p in model.named_parameters()
        if p.requires_grad
    }

    model.train()
    iterator = iter(train_loader)

    valid_avg_steps = 0

    for step in tqdm(range(avg_steps), desc="Post-convergence SGD averaging"):
        try:
            x, y = next(iterator)
        except StopIteration:
            iterator = iter(train_loader)
            x, y = next(iterator)

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = F.cross_entropy(logits, y)

        if not torch.isfinite(loss):
            print("NaN/Inf detected during post-convergence averaging. Stop averaging.")
            break

        loss.backward()
        optimizer.step()

        with torch.no_grad():
            for name, p in model.named_parameters():
                if p.requires_grad:
                    avg_state[name] += p.data

        valid_avg_steps += 1

    if valid_avg_steps > 0:
        avg_model = copy.deepcopy(model).to(device)
        with torch.no_grad():
            for name, p in avg_model.named_parameters():
                if name in avg_state:
                    p.data.copy_(avg_state[name] / valid_avg_steps)
    else:
        print("Averaging was skipped. Using last valid model.")
        avg_model = copy.deepcopy(model).to(device)

    alpha_df, mean_alpha = estimate_model_tail_indices_mmo(avg_model)
    mean_alpha_prunable = mean_alpha_prunable_layers_from_df(alpha_df, skip_last=True)
    final_test = evaluate(avg_model, test_loader)

    train_time_sec = time.time() - start_time

    actual_epochs = history[-1]["epoch"] if len(history) > 0 else 0

    summary = {
        **asdict(config),
        "regime": "SGD",
        "optimizer": "SGD_no_momentum_constant_lr",
        "eta_over_b": config.lr / config.batch_size,
        "n_params": count_parameters(avg_model),
        "max_epochs": max_epochs,
        "actual_epochs": actual_epochs,
        "last_valid_epoch": last_valid_epoch,
        "last_valid_train_loss": np.nan if last_valid_row is None else last_valid_row["train_loss"],
        "last_valid_train_acc": np.nan if last_valid_row is None else last_valid_row["train_acc"],
        "last_valid_test_acc": np.nan if last_valid_row is None else last_valid_row["test_acc"],
        "converged_by_article_criterion": converged,
        "stopped_by_nan": stopped_by_nan,
        "avg_steps_requested": avg_steps,
        "avg_steps_used": valid_avg_steps,
        "final_test_acc": final_test["acc"],
        "final_test_loss": final_test["loss"],
        "mean_alpha": mean_alpha,
        "mean_alpha_prunable_layers": mean_alpha_prunable,
        "alpha_estimator": "Barsbey-style multidimensional MMO",
        "train_time_sec": train_time_sec
    }

    return avg_model, pd.DataFrame(history), alpha_df, summary


In [ ]:
# Запуск серии SGD-экспериментов

# lr — шаг обучения eta, batch_size — размер мини-батча b.
# Заданы 24 конфигурации для двух размеров мини-батча: b=100 и b=50.
# При нехватке времени в Colab можно временно оставить только часть конфигураций.

SGD_SWEEP_SPECS = [
    # Конфигурации для b = 100
    {"lr": 0.005, "batch_size": 100},   # eta/b = 0.00005
    {"lr": 0.025, "batch_size": 100},   # eta/b = 0.00025
    {"lr": 0.050, "batch_size": 100},   # eta/b = 0.00050
    {"lr": 0.100, "batch_size": 100},   # eta/b = 0.00100
    {"lr": 0.150, "batch_size": 100},   # eta/b = 0.00150
    {"lr": 0.200, "batch_size": 100},   # eta/b = 0.00200
    {"lr": 0.250, "batch_size": 100},   # eta/b = 0.00250
    {"lr": 0.310, "batch_size": 100},   # eta/b = 0.00310

    # Конфигурации для b = 50
    {"lr": 0.025, "batch_size": 50},    # eta/b = 0.00050
    {"lr": 0.050, "batch_size": 50},    # eta/b = 0.00100
    {"lr": 0.075, "batch_size": 50},    # eta/b = 0.00150
    {"lr": 0.100, "batch_size": 50},    # eta/b = 0.00200
    {"lr": 0.150, "batch_size": 50},    # eta/b = 0.00300
    {"lr": 0.200, "batch_size": 50},    # eta/b = 0.00400
    {"lr": 0.250, "batch_size": 50},    # eta/b = 0.00500
    {"lr": 0.300, "batch_size": 50},    # eta/b = 0.00600
    {"lr": 0.350, "batch_size": 50},    # eta/b = 0.00700
    {"lr": 0.400, "batch_size": 50},    # eta/b = 0.00800
    {"lr": 0.430, "batch_size": 50},    # eta/b = 0.00860
    {"lr": 0.460, "batch_size": 50},    # eta/b = 0.00920
    {"lr": 0.500, "batch_size": 50},    # eta/b = 0.01000
    {"lr": 0.520, "batch_size": 50},    # eta/b = 0.01040
    {"lr": 0.550, "batch_size": 50},    # eta/b = 0.01100
    {"lr": 0.570, "batch_size": 50},    # eta/b = 0.01140
]

# Чтобы быстро проверить код, можно поставить True.
# Для финального дипломного запуска лучше False.
FAST_DEBUG = False

if FAST_DEBUG:
    MAX_EPOCHS = 1
    AVG_STEPS = 10
    SGD_SWEEP_SPECS = [
        {"lr": 0.10, "batch_size": 100}
    ]
else:
    MAX_EPOCHS = 80
    AVG_STEPS = 1000

rows = []
all_alpha_rows = []

for sweep_id, spec in enumerate(SGD_SWEEP_SPECS):
    cfg = SGDConfig(
        width=2048,
        lr=spec["lr"],
        batch_size=spec["batch_size"],
        seed=42 + sweep_id
    )

    print("\n" + "=" * 80)
    print("SGD sweep:", sweep_id, asdict(cfg))
    print("=" * 80)

    try:
        model, hist_df, alpha_df, summary = train_sgd_article_style(
            cfg,
            max_epochs=MAX_EPOCHS,
            convergence_train_acc=0.99,
            convergence_loss=0.03,
            avg_steps=AVG_STEPS,
            verbose=True
        )

        summary["sweep_id"] = sweep_id
        summary["status"] = "ok"
        rows.append(summary)

        # Save history, alpha by layers, checkpoint
        prefix = f"sgd_sweep_{sweep_id}_b{cfg.batch_size}_lr{cfg.lr}".replace(".", "p")

        hist_path = os.path.join(RESULTS_DIR, f"{prefix}_history.csv")
        alpha_path = os.path.join(RESULTS_DIR, f"{prefix}_alpha_layers.csv")
        ckpt_path = os.path.join(CHECKPOINTS_DIR, f"{prefix}.pt")

        hist_df.to_csv(hist_path, index=False)
        alpha_df.to_csv(alpha_path, index=False)

        torch.save({
            "model_state_dict": model.state_dict(),
            "summary": summary,
            "alpha_df": alpha_df.to_dict(orient="records"),
            "history": hist_df.to_dict(orient="records")
        }, ckpt_path)

        alpha_df2 = alpha_df.copy()
        alpha_df2["sweep_id"] = sweep_id
        alpha_df2["lr"] = cfg.lr
        alpha_df2["batch_size"] = cfg.batch_size
        alpha_df2["eta_over_b"] = cfg.lr / cfg.batch_size
        all_alpha_rows.append(alpha_df2)

        print("Saved checkpoint:", ckpt_path)

    except Exception as e:
        print("ERROR:", repr(e))
        rows.append({
            "sweep_id": sweep_id,
            "status": "error",
            "lr": spec["lr"],
            "batch_size": spec["batch_size"],
            "eta_over_b": spec["lr"] / spec["batch_size"],
            "error": repr(e)
        })

summary_df = pd.DataFrame(rows)
summary_df = summary_df.sort_values(["batch_size", "eta_over_b"]).reset_index(drop=True)
summary_df.to_csv(os.path.join(RESULTS_DIR, "sgd_sweep_summary.csv"), index=False)

if len(all_alpha_rows) > 0:
    alpha_all_df = pd.concat(all_alpha_rows, ignore_index=True)
    alpha_all_df.to_csv(os.path.join(RESULTS_DIR, "sgd_alpha_by_layer_all.csv"), index=False)

display(summary_df)


In [ ]:
# График зависимости хвостового показателя от отношения eta/b

df = pd.read_csv(os.path.join(RESULTS_DIR, "sgd_sweep_summary.csv"))

valid = df[(df["status"] == "ok") & df["mean_alpha"].notna()].copy()
valid = valid.sort_values("eta_over_b")

display(valid[[
    "sweep_id", "width", "lr", "batch_size", "eta_over_b",
    "final_test_acc", "mean_alpha", "mean_alpha_prunable_layers",
    "converged_by_article_criterion", "n_params", "train_time_sec"
]])

plt.figure(figsize=(7, 5))

for b, g in valid.groupby("batch_size"):
    g = g.sort_values("eta_over_b")

    plt.scatter(
        g["eta_over_b"],
        g["mean_alpha"],
        s=55,
        marker="o",
        color="tab:blue",        # все точки синие
        edgecolor="black",
        linewidth=0.5,
        label=f"b={int(b)}"
    )

plt.xscale("log")
plt.xlabel(r"$\eta/b$")
plt.ylabel(r"mean $\widehat{\alpha}$")
plt.title("FCN4 / MNIST: tail index vs eta/b")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.tight_layout()

fig_path = os.path.join(FIGURES_DIR, "sgd_alpha_vs_eta_over_b.png")
plt.savefig(fig_path, dpi=200)
plt.show()

print("Saved:", fig_path)

In [ ]:
# Разреживание сохранённых SGD-моделей

RUN_PRUNING = True
RUN_SVD_PRUNING = True   # Если вычисление SVD занимает слишком много времени, можно поставить False.

PRUNE_SKIP_LAST_WEIGHT_LAYER = True
PRUNING_KEEP_RATIOS = [1.0, 0.8, 0.6, 0.4, 0.2, 0.1]


def get_weight_parameter_names(model, skip_last=True):
    names = [
        name
        for name, p in model.named_parameters()
        if ("weight" in name and p.ndim >= 2)
    ]
    if skip_last and len(names) > 1:
        names = names[:-1]
    return names


def load_sgd_checkpoint_model(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    summary = ckpt["summary"]

    model = build_fcn4(width=int(summary.get("width", 2048))).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    return model, summary, ckpt


def apply_global_magnitude_pruning(model, keep_ratio, skip_last=True):
    pruned = copy.deepcopy(model).to(device)
    names = get_weight_parameter_names(pruned, skip_last=skip_last)

    all_scores = []
    with torch.no_grad():
        for name, p in pruned.named_parameters():
            if name in names:
                all_scores.append(p.detach().abs().flatten())

        scores = torch.cat(all_scores)
        total = scores.numel()
        k = int(math.ceil(keep_ratio * total))

        if k >= total:
            return pruned
        if k <= 0:
            threshold = torch.inf
        else:
            threshold = torch.topk(scores, k, largest=True).values.min()

        for name, p in pruned.named_parameters():
            if name in names:
                mask = (p.detach().abs() >= threshold).to(p.dtype)
                p.mul_(mask)

    return pruned


def apply_layerwise_magnitude_pruning(model, keep_ratio, skip_last=True):
    pruned = copy.deepcopy(model).to(device)
    names = get_weight_parameter_names(pruned, skip_last=skip_last)

    with torch.no_grad():
        for name, p in pruned.named_parameters():
            if name not in names:
                continue

            scores = p.detach().abs().flatten()
            total = scores.numel()
            k = int(math.ceil(keep_ratio * total))

            if k >= total:
                continue
            if k <= 0:
                threshold = torch.inf
            else:
                threshold = torch.topk(scores, k, largest=True).values.min()

            mask = (p.detach().abs() >= threshold).to(p.dtype)
            p.mul_(mask)

    return pruned


def apply_node_pruning(model, keep_ratio, skip_last=True):
    """
    Обнуляет столбцы матриц весов с малыми l2-нормами.
    Для Linear слоя W имеет размер out_features x in_features.
    Столбец соответствует входному нейрону/каналу данного слоя.
    """
    pruned = copy.deepcopy(model).to(device)
    names = get_weight_parameter_names(pruned, skip_last=skip_last)

    with torch.no_grad():
        for name, p in pruned.named_parameters():
            if name not in names:
                continue

            W = p.data
            if W.ndim != 2:
                continue

            col_norms = torch.linalg.norm(W, ord=2, dim=0)
            total_cols = col_norms.numel()
            k = int(math.ceil(keep_ratio * total_cols))

            if k >= total_cols:
                continue
            if k <= 0:
                mask_cols = torch.zeros_like(col_norms, dtype=torch.bool)
            else:
                threshold = torch.topk(col_norms, k, largest=True).values.min()
                mask_cols = col_norms >= threshold

            W.mul_(mask_cols.to(W.dtype).view(1, -1))

    return pruned


def apply_svd_pruning(model, keep_ratio, skip_last=True):
    """
    Для каждой матрицы W строит низкоранговое приближение через SVD.
    Сохраняется ceil(keep_ratio * rank_max) сингулярных чисел.
    """
    pruned = copy.deepcopy(model).to(device)
    names = get_weight_parameter_names(pruned, skip_last=skip_last)

    with torch.no_grad():
        for name, p in pruned.named_parameters():
            if name not in names:
                continue

            W = p.data
            if W.ndim != 2:
                continue

            # SVD лучше считать в float32/float64 без градиентов
            U, S, Vh = torch.linalg.svd(W, full_matrices=False)
            r_max = S.numel()
            r = int(math.ceil(keep_ratio * r_max))

            if r >= r_max:
                continue
            if r <= 0:
                W.zero_()
            else:
                W_approx = (U[:, :r] * S[:r].view(1, -1)) @ Vh[:r, :]
                W.copy_(W_approx)

    return pruned


def evaluate_pruning_for_checkpoint(ckpt_path, test_loader, keep_ratios, skip_last=True):
    base_model, summary, ckpt = load_sgd_checkpoint_model(ckpt_path)
    base_eval = evaluate(base_model, test_loader)
    base_acc = base_eval["acc"]

    methods = [
        ("global_magnitude", apply_global_magnitude_pruning),
        ("layerwise_magnitude", apply_layerwise_magnitude_pruning),
        ("node_pruning", apply_node_pruning),
    ]

    if RUN_SVD_PRUNING:
        methods.append(("svd_pruning", apply_svd_pruning))

    rows = []

    for method_name, method_fn in methods:
        for keep_ratio in keep_ratios:
            pruned_model = method_fn(base_model, keep_ratio, skip_last=skip_last)
            metrics = evaluate(pruned_model, test_loader)

            rows.append({
                "checkpoint": os.path.basename(ckpt_path),
                "method": method_name,
                "keep_ratio": keep_ratio,
                "pruned_fraction": 1.0 - keep_ratio,
                "base_test_acc": base_acc,
                "pruned_test_acc": metrics["acc"],
                "relative_test_acc": metrics["acc"] / base_acc if base_acc > 0 else np.nan,
                "lr": summary.get("lr", np.nan),
                "batch_size": summary.get("batch_size", np.nan),
                "eta_over_b": summary.get("eta_over_b", np.nan),
                "mean_alpha": summary.get("mean_alpha", np.nan),
                "mean_alpha_prunable_layers": summary.get("mean_alpha_prunable_layers", np.nan),
                "final_test_acc": summary.get("final_test_acc", np.nan),
                "actual_epochs": summary.get("actual_epochs", np.nan),
                "converged_by_article_criterion": summary.get("converged_by_article_criterion", np.nan),
            })

            del pruned_model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    return pd.DataFrame(rows)


if RUN_PRUNING:
    # Используем test_loader с любым batch size, он нужен только для оценки качества.
    _, pruning_test_loader, _, _ = get_dataloaders(
        dataset="MNIST",
        batch_size=256,
        shuffle_train=False
    )

    ckpt_paths = sorted(glob.glob(os.path.join(CHECKPOINTS_DIR, "sgd_sweep_*.pt")))

    if len(ckpt_paths) == 0:
        print("No SGD checkpoints found. Сначала запусти SGD sweep.")
    else:
        print("Found checkpoints:")
        for p in ckpt_paths:
            print("  ", p)

        pruning_dfs = []
        for ckpt_path in ckpt_paths:
            print("\nPruning checkpoint:", ckpt_path)
            df_pr = evaluate_pruning_for_checkpoint(
                ckpt_path,
                pruning_test_loader,
                PRUNING_KEEP_RATIOS,
                skip_last=PRUNE_SKIP_LAST_WEIGHT_LAYER
            )
            pruning_dfs.append(df_pr)

        pruning_all_df = pd.concat(pruning_dfs, ignore_index=True)
        pruning_csv_path = os.path.join(RESULTS_DIR, "sgd_pruning_all_models.csv")
        pruning_all_df.to_csv(pruning_csv_path, index=False)

        display(pruning_all_df.sort_values(["method", "eta_over_b", "keep_ratio"]))

        # Краткая таблица: при какой alpha и keep_ratio сохраняется >= 95% исходной точности.
        stable = pruning_all_df[pruning_all_df["relative_test_acc"] >= 0.95].copy()
        if len(stable) > 0:
            stable_summary = (
                stable.groupby(["checkpoint", "method", "eta_over_b", "mean_alpha"], as_index=False)
                .agg(min_keep_ratio_for_95pct=("keep_ratio", "min"))
                .sort_values(["method", "mean_alpha"])
            )
            stable_summary.to_csv(os.path.join(RESULTS_DIR, "sgd_pruning_95pct_summary.csv"), index=False)
            print("Минимальная доля сохранённых параметров для сохранения >=95% исходной точности:")
            display(stable_summary)

        # Графики relative accuracy vs keep_ratio для каждого метода.
        # Цвета сделаны в логике рисунков статьи:
        # меньшая alpha -> более тёмный сине-зелёный цвет;
        # большая alpha -> более светлый жёлтый цвет.
        alpha_min = pruning_all_df["mean_alpha"].min()
        alpha_max = pruning_all_df["mean_alpha"].max()
        norm = plt.Normalize(vmin=alpha_min, vmax=alpha_max)
        cmap = plt.get_cmap("YlGnBu_r")

        for method in pruning_all_df["method"].unique():
            sub = pruning_all_df[pruning_all_df["method"] == method].copy()

            plt.figure(figsize=(7, 5))
            ax = plt.gca()

            # Сортируем по alpha: светлые кривые рисуются первыми,
            # тёмные и более тяжёлохвостые остаются видимыми сверху.
            grouped = list(sub.groupby(["checkpoint", "mean_alpha", "eta_over_b"]))
            grouped = sorted(grouped, key=lambda item: item[0][1], reverse=True)

            for (ckpt_name, alpha, eta_over_b), g in grouped:
                g = g.sort_values("keep_ratio")
                color = cmap(norm(alpha))
                label = rf"$\widehat{{a}}={alpha:.3f}$, $\eta/b={eta_over_b:.4g}$"
                ax.plot(
                    g["keep_ratio"],
                    g["relative_test_acc"],
                    marker="o",
                    linewidth=1.6,
                    markersize=4,
                    color=color,
                    label=label
                )

            ax.set_xlabel("Доля сохранённых параметров")
            ax.set_ylabel("Относительная test accuracy")
            ax.set_title(f"Pruning curve: {method}")
            ax.grid(True, alpha=0.3)
            ax.set_ylim(0.0, 1.05)

            sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
            sm.set_array([])
            cbar = plt.colorbar(sm, ax=ax)
            cbar.set_label(r"mean $\widehat{a}$")

            # При 20+ моделях легенда становится слишком большой.
            # Сохраняем отдельную csv-таблицу, а на графике оставляем цветовую шкалу.
            if len(grouped) <= 12:
                ax.legend(fontsize=7)

            plt.tight_layout()

            fig_path = os.path.join(FIGURES_DIR, f"pruning_{method}.png")
            plt.savefig(fig_path, dpi=200)
            plt.show()

        print("Saved pruning results:", pruning_csv_path)


In [ ]:
# Архивация результатов эксперимента

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_name = f"vkr_results_{timestamp}.zip"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for d in [RESULTS_DIR, FIGURES_DIR, CHECKPOINTS_DIR]:
        if not os.path.exists(d):
            continue
        for root, _, files in os.walk(d):
            for fn in files:
                path = os.path.join(root, fn)
                zf.write(path, arcname=path)

print("Created:", zip_name)

try:
    from google.colab import files
    files.download(zip_name)
except Exception:
    print("Если это не Colab, файл лежит в текущей папке:", zip_name)